# 1 — Load a protein into mBuild and export it

mBuild's `biopolymers` module loads a protonated protein PDB by **template
matching** against Chemical Component Dictionary residues: every atom is
identified, bonds carry orders, residues carry formal charges and real PDB
numbers, and anything unmatched raises an error naming the residue and the
fix — chemistry is never guessed.

From that one Compound, the same structure exports to **GMSO**, **ParmEd**,
**RDKit**, a prepared **PDB**, and (notebook 2) the **OpenFF** ecosystem.


In [1]:
# One-time preparation: protonate the crystal structure at pH 7.
import os

if not os.path.exists("1ubq_protonated.pdb"):
    from pdbfixer import PDBFixer
    from openmm.app import PDBFile

    fixer = PDBFixer(filename="1UBQ_testProtein.cleaned.pdb")
    fixer.findMissingResidues(); fixer.missingResidues = {}
    fixer.findNonstandardResidues(); fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(keepWater=False)
    fixer.findMissingAtoms(); fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    with open("1ubq_protonated.pdb", "w") as handle:
        PDBFile.writeFile(fixer.topology, fixer.positions, handle)


In [2]:
from mbuild.biopolymers import Protein

protein = Protein("1ubq_protonated.pdb")
print(len(list(protein.residues())), "residues |", protein.n_particles,
      "atoms | net formal charge:", protein.net_formal_charge)
lys = protein.get_residue(48, chain_id="A")
print("residue LYS 48: formal charge", lys.formal_charge,
      "| atoms:", [p.name for p in lys.particles()][:6], "...")


76 residues | 1231 atoms | net formal charge: 0
residue LYS 48: formal charge 1 | atoms: ['N', 'H', 'CA', 'HA', 'C', 'O'] ...


## Export to GMSO

Residue names, **real PDB residue numbers**, and chain labels arrive intact —
the metadata GMSO workflows key on for per-molecule force-field application
and template mapping.


In [3]:
topology = protein.to_gmso()
site = next(iter(topology.sites))
print(topology.n_sites, "sites |", topology.n_bonds, "bonds")
print("first site:", site.name, "| residue:", site.residue.name,
      site.residue.number, "| chain:", site.molecule.name)


1231 sites | 1237 bonds
first site: N | residue: MET 1 | chain: Chain_A


## Export to ParmEd, RDKit, and PDB


In [4]:
structure = protein.to_parmed()          # residues intact
print("ParmEd:", len(structure.residues), "residues")

rdmol = protein.to_rdkit()               # sanitized; charges + residue info
from rdkit import Chem
print("RDKit:", rdmol.GetNumAtoms(), "atoms | net formal charge:",
      Chem.GetFormalCharge(rdmol))

protein.save_pdb("1ubq_prepared.pdb", overwrite=True)


ParmEd: 76 residues
RDKit: 1231 atoms | net formal charge: 0


## Export to OpenFF

The prepared PDB loads directly through OpenFF Pablo — for an unmodified
protein, no extra information is needed at all.


In [5]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

off_topology = topology_from_pdb("1ubq_prepared.pdb",
                                 residue_library=STD_CCD_CACHE)
molecule = off_topology.molecule(0)
print("OpenFF:", molecule.n_atoms, "atoms | net charge:",
      molecule.total_charge)


OpenFF: 1231 atoms | net charge: 0.0 elementary_charge


**Next**: notebook 2 modifies this protein covalently and takes it all
the way to a solvated MD simulation.
